# SageMaker: Train and Deploy the Fashion-MNIST CNN

**Run this notebook from the repo root** (same folder as `sagemaker/` and `src/`).

Run the cells below **in order**, one at a time (Shift+Enter). Don't use "Run All" the first
time — Cell 2 (training) takes a few minutes and it's worth watching the logs before moving on.


## 1. Setup

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()  # auto-created S3 bucket for this account/region
print("Using bucket:", bucket)
print("Using role:", role)


## 2. Launch the training job

This blocks and streams training logs until the job finishes (a few minutes on `ml.m5.large`). `train.py` downloads Fashion-MNIST itself inside the training container.

In [ ]:
estimator = PyTorch(
    entry_point="train.py",
    source_dir="sagemaker",
    dependencies=["src"],             # bundles src/ alongside the entry point
    role=role,
    framework_version="2.1",
    py_version="py310",
    instance_type="ml.m5.large",      # CPU is enough for this model; no need for a GPU instance
    instance_count=1,
    hyperparameters={
        "model": "cnn",
        "kernel-size": 3,
        "epochs": 8,
        "batch-size": 256,
        "lr": 0.001,
    },
    output_path=f"s3://{bucket}/fashion-mnist-cnn/output",
)

estimator.fit()


## 3. Deploy to a real-time endpoint

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    entry_point="inference.py",
    source_dir="sagemaker",
)
print("Endpoint name:", predictor.endpoint_name)


## 4. Test the endpoint with one real test image

In [ ]:
import json
from torchvision import datasets, transforms

test_set = datasets.FashionMNIST(root="/tmp/data", train=False, download=True, transform=transforms.ToTensor())
image, true_label = test_set[0]

predictor.serializer = sagemaker.serializers.JSONSerializer()
predictor.deserializer = sagemaker.deserializers.JSONDeserializer()

response = predictor.predict({"image": image.squeeze(0).tolist()})
print("True label index:", true_label)
print("Prediction:", response)


## 5. Clean up

**Run this before closing the notebook** — the endpoint bills per hour while it exists.

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted.")
